# Introduction to Meridian Demo (Local, TensorFlow Backend)

This notebook walks through the core Meridian workflow, adapted to run locally (no Colab/Google Drive dependencies):

0. Install and environment configuration
1. Load the data
2. Configure the model
3. Run exploratory data analysis & two-pager output
4. Fit the model
5. Run post-modeling health checks
6. Run model diagnostics
7. Generate model results & two-page output
8. Run budget optimization & two-page output
9. Save the model object

Note: this notebook uses the sample simulated dataset, so results won't reflect a real dataset. A GPU is strongly recommended for fitting the model in reasonable time, but not required.

## Step 0: Install and Environment Configuration

Run the cell below once to install Meridian if it isn't already installed in your local environment. Uncomment the line matching your setup.

In [ ]:
# Install meridian: from PyPI @ latest release\n!uv pip install --upgrade google-meridian[and-cuda,schema]\n\n# Install meridian: from PyPI @ specific version\n# !uv pip install google-meridian[and-cuda,schema]==1.3.1\n\n# Install meridian: from GitHub @HEAD\n# !uv pip install --upgrade "google-meridian[and-cuda,schema] @ git+https://github.com/google/meridian.git@main"

In [1]:
import os
os.environ['MERIDIAN_BACKEND'] = 'tensorflow'

In [ ]:
# from dotenv import load_dotenv
# load_dotenv("gpu.env")

In [2]:
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.model.eda import meridian_eda
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Available RAM: {:.1f} GB\n'.format(ram_gb))
print('Num GPUs Available: ', len(tf.config.experimental.list_physical_devices('GPU')))
print('Num CPUs Available: ', len(tf.config.experimental.list_physical_devices('CPU')))

I0000 00:00:1789137941.496536   34360 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789137941.874080   34360 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789137944.334647   34360 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Available RAM: 16.5 GB

Num GPUs Available:  1
Num CPUs Available:  1


W0000 00:00:1789137947.143603   34360 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


### Local output directory

All generated reports and the saved model will be written to `meridian_root`, a local folder. Change this path if you'd like outputs saved elsewhere.

In [3]:
meridian_root = os.path.join(os.getcwd(), 'meridian_output')
os.makedirs(meridian_root, exist_ok=True)
print(f'Outputs will be saved to: {meridian_root}')

Outputs will be saved to: /home/smenon01/repos/meridian-ai/demo_notebooks/meridian_output


## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_all_channels.csv).

In [4]:
df = pd.read_csv(
    # To use a local file instead, replace with a path such as "./data/geo_all_channels.csv"
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv",
    index_col=0,
)

### Dataset preview and schema overview

A preview of the aggregated dataset is shown below. For details on Meridian's data requirements, see [Collect and organize your data](https://developers.google.com/meridian/docs/pre-modeling/collect-data).

In [5]:
preview_df = df[df['time'].isin(['2021-01-25', '2021-02-01', '2021-02-08'])].head(6)
preview_df

,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,competitor_sales_control,sentiment_score_control,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,Promo,conversions,revenue_per_conversion,population
0,Geo0,2021-01-25,280668,0,0,470611,108010,-1.338765,0.115581,2058.06080,0.0000,0.00000,3667.3965,841.604400,97320,0.000000,1954576.8,0.020055,136670.94
1,Geo0,2021-02-01,366206,182108,19825,527702,252506,0.893645,0.944224,2685.28740,1755.7454,147.31808,4112.2974,1967.504400,201441,0.000000,2064249.6,0.020103,136670.94
2,Geo0,2021-02-08,197565,230170,0,393618,184061,-0.284549,-1.290579,1448.68950,2219.1223,0.00000,3067.4023,1434.187000,0,0.683819,2086382.8,0.019929,136670.94
156,Geo1,2021-01-25,293884,191662,108120,415415,710474,0.035612,-0.947813,2154.97000,1847.8578,803.43160,3237.2627,5535.950700,0,0.000000,2126297.8,0.020085,199816.90
157,Geo1,2021-02-01,597519,360999,255145,584855,452186,1.009433,1.322983,4381.44140,3480.4750,1895.96330,4557.6816,3523.393300,236553,0.000000,1900217.6,0.019933,199816.90
158,Geo1,2021-02-08,101282,456699,0,346725,5589,-1.590099,-1.359786,742.67285,4403.1410,0.00000,2701.9724,43.548992,0,0.000000,7177179.0,0.020159,199816.90


**Dataset legend:**

| Column | Variable type | Description |
| --- | --- | --- |
| time | Time | The specific time period for the data row (for example, weekly). |
| geo | Geography | The geographic region for the observation (for example, DMA, state). |
| conversions | KPI | The model's response variable, for example total sales or sign-ups. |
| revenue_per_conversion | Revenue per KPI | The average revenue generated per KPI unit. |
| Channel0_spend, Channel1_spend, Channel2_spend, Channel3_spend, Channel4_spend | Media spend | Total media spend for each paid channel during that week and geo. |
| Channel0_impression, ..., Channel4_impression | Media data | The media execution metric (e.g. impressions, clicks) for those paid channels. |
| Organic_channel0_impression | Organic media | An organic media metric that drives outcomes but has no direct media cost. |
| competitor_sales_control, sentiment_score_control, Promo | Control variables | Confounders that affect both your outcome and your media decisions. |
| population | Population | The population of the geographic region. |

Create a `DataFrameInputDataBuilder` instance and offer the components to the builder (they may be offered all at once or piecewise).

In [6]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

In [7]:
builder = (
    builder.with_kpi(df)
    .with_revenue_per_kpi(df)
    .with_population(df)
    .with_controls(
        df, control_cols=["sentiment_score_control", "competitor_sales_control"]
    )
)

channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
)

If your data includes organic media or non-media treatments, add them using `with_organic_media` and `with_non_media_treatments`. See [Collect and organize your data](https://developers.google.com/meridian/docs/pre-modeling/collect-data) for variable definitions.

In [8]:
builder = builder.with_non_media_treatments(
    df, non_media_treatment_cols=['Promo']
).with_organic_media(
    df,
    organic_media_cols=['Organic_channel0_impression'],
    organic_media_channels=['Organic_channel0'],
)

Finally, build the `InputData`.

In [9]:
data = builder.build()

Note: this simulated data does not contain reach and frequency. We recommend including reach and frequency whenever available -- see [Bayesian Hierarchical Media Mix Model Incorporating Reach and Frequency Data](https://research.google/pubs/bayesian-hierarchical-media-mix-model-incorporating-reach-and-frequency-data/). For loading reach/frequency data, see [Load geo-level data with reach and frequency](https://developers.google.com/meridian/docs/user-guide/load-geo-data-with-rf), and for other formats see [Supported data types and formats](https://developers.google.com/meridian/docs/user-guide/supported-data-types-formats).

## Step 2: Configure the model

Meridian uses a Bayesian framework and MCMC to sample from the posterior distribution.

Initialize the `Meridian` class with the loaded data and a model specification. One advantage of Meridian is its capacity to calibrate the model directly through ROI priors, as described in [Media Mix Model Calibration With Bayesian Priors](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/). Here, the ROI prior for all media channels is identical: Lognormal(0.2, 0.9).

In [10]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, enable_aks=True)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

W0000 00:00:1789132997.792621   17061 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1789132998.099492   17061 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9226 MB memory:  -> device: 0, name: NVIDIA RTX PRO 3000 Blackwell Generation Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
I0000 00:00:1789133001.275968   17061 service.cc:153] XLA service 0x403a5870 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789133001.275998   17061 service.cc:161]   StreamExecutor [0]: NVIDIA RTX PRO 3000 Blackwell Generation Laptop GPU, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.26.0)
I0000 00:00:1789133001.305603   17061 cuda_dnn.cc:461] Loaded cuDNN version 92600
I0000 00:00:1789133001.406213   1706

Use `sample_prior()` to obtain samples from the prior distributions of model parameters.

In [11]:
mmm.sample_prior(500)

/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/model/knots.py:614: RuntimeWarning: overflow encountered in cast
  backend.np_float_dtype(math.comb(ncol, design_mat.shape[1]))


## Step 3: Run exploratory data analysis & two-pager output

Meridian EDA helps identify potential issues with your dataset before fitting the model. To obtain the EDA HTML report, initialize `MeridianEDA` with the `Meridian` model object and call `generate_and_save_report`.

Note: `MeridianEDA` requires prior samples to run. It will automatically run prior sampling if the model hasn't already done so.

In [12]:
eda = meridian_eda.MeridianEDA(mmm)
filename = 'eda_report.html'
eda.generate_and_save_report(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=os.path.join(meridian_root, filename))

I0000 00:00:1789133168.368044   17061 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


For more on Meridian's EDA checks, visualizations, and customizations, see [Meridian's EDA package](https://developers.google.com/meridian/docs/pre-modeling/perform-eda#meridians_eda_package).

## Step 4: Fit the model

Use `sample_posterior()` to obtain samples from the posterior distributions of model parameters. This can take several minutes to tens of minutes depending on your hardware (GPU recommended).

In [13]:
%%time
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

W0000 00:00:1789133315.933304   17061 assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert
I0000 00:00:1789133383.078760   17061 dot_merger.cc:481] Merging Dots in computation: body_wrapper.1824.clone.sunk.clone.clone
I0000 00:00:1789133383.084465   17061 dot_merger.cc:481] Merging Dots in computation: a_inference__tf_xla_windowed_adaptive_nuts_122401__XlaMustCompile_true_config_proto_2201667018877855759_executor_type_11160318154034397263_.1917
I0000 00:00:1789133387.739721   17061 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1789133388.127350   19140 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_Einsum_320', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1789133

CPU times: user 6min 40s, sys: 39.8 s, total: 7min 20s
Wall time: 7min 6s


/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


For configuring parameters and customized model specs (e.g. different ROI priors per channel), see [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model).

## Step 5: Run post-modeling health checks

These checks diagnose common issues related to model convergence, specification, and plausibility.

In [14]:
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=os.path.join(meridian_root, filename))

/tmp/ipykernel_17061/871727528.py:1: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()


Metric check,Status,Recommended action
Convergence,Fail,"The model hasn't converged, and the `max_r_hat` for parameter `mu_t` is inf. We recommend increasing MCMC iterations or investigating model misspecification (e.g., priors, multicollinearity) before proceeding."


## Step 6: Run model diagnostics

To further assess convergence and model fit, use the `visualizer` module.

1. Assess convergence: R-hat close to 1.0 indicates convergence; R-hat < 1.2 is a reasonable threshold for many problems.

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

2. Assess the model's fit by comparing expected vs. actual sales.

In [15]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

I0000 00:00:1789135547.409379   27099 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1789135547.416433   27105 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1789135547.418115   27107 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 16 bytes spill stores, 16 bytes spill loads



alt.LayerChart(...)

For more diagnostics, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

## Step 7: Generate model results & two-page output

Initialize the `Summarizer` class with the model object, then export the two-page HTML summary for a given time range.

In [16]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [17]:
start_date = '2021-01-25'
end_date = '2024-01-15'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', meridian_root, start_date, end_date
)

/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/analysis/analyzer.py:2609: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(


Preview of the two-page output based on the simulated data:

In [18]:
IPython.display.HTML(filename=os.path.join(meridian_root, 'summary_output.html'))

Dataset,R-squared,MAPE,wMAPE
All Data,-146.41,80%,79%


For a customized report, results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).

## Step 8: Run budget optimization & generate an optimization report

You can choose which scenario to run for budget allocation. The default scenario finds the optimal allocation across channels for a given budget to maximize ROI.

For a sharable interactive dashboard, see [Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner).

Instantiate `BudgetOptimizer` and run `optimize()` with no customization to use the default Fixed Budget Scenario to maximize ROI.

In [19]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

CPU times: user 20.9 s, sys: 1.12 s, total: 22 s
Wall time: 25.8 s


Export the two-page HTML optimization report, containing optimized spend allocations and ROI.

In [20]:
optimization_results.output_optimization_summary(
    'optimization_output.html', meridian_root
)

In [21]:
IPython.display.HTML(filename=os.path.join(meridian_root, 'optimization_output.html'))

Channel,Non-optimized spend,Optimized spend
Channel3,40%,31%
Channel4,22%,28%
Channel0,18%,24%
Channel1,14%,10%
Channel2,6%,7%


For customized optimization scenarios (e.g. flexible budgets), see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more on results/visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).

Optimization can also be performed on a hypothetical future-scenario dataset, with the same structure as the input data, encoding an anticipated flighting pattern, cost per media unit, and revenue per KPI.

In [24]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv"
)

The new data is converted into a set of multi-dimensional arrays and used to construct a `DataTensors` instance, which is passed to `optimize()` as `new_data`.

Constructing a `DataTensors` requires that all arrays have "time" and "geo" dimensions. Alternatively, `BudgetOptimizer.create_optimization_tensors` can construct a `DataTensors` instance, which can be convenient when constant values (e.g. revenue per kpi) are assumed across geos/time.

In [25]:
n_geos = mmm.n_geos
n_media_channels = mmm.n_media_channels
n_non_media_channels = mmm.n_non_media_channels
n_organic_media_channels = mmm.n_organic_media_channels

# The number of time periods and time range do not need to match the input data.
df[constants.TIME] = pd.to_datetime(df[constants.TIME], errors='coerce')
unique_times = sorted(df[constants.TIME].unique())
n_times = len(unique_times)

geos = mmm.input_data.geo.values
media_channels = mmm.input_data.media_channel.values
media_cols = [f"{channel}_impression" for channel in media_channels]
media_spend_cols = [f"{channel}_spend" for channel in media_channels]
non_media_treatment_cols = ['Promo']
organic_media_cols = ['Organic_channel0_impression']
organic_media_channels = ['Organic_channel0']
revenue_per_kpi_col = 'revenue_per_conversion'
times_str = [time.strftime(constants.DATE_FORMAT) for time in unique_times]

media_np = np.zeros((n_geos, n_times, n_media_channels))
media_spend_np = np.zeros((n_geos, n_times, n_media_channels))
non_media_treatment_np = np.zeros((n_geos, n_times, n_non_media_channels))
organic_media_np = np.zeros((n_geos, n_times, n_organic_media_channels))
revenue_per_kpi_np = np.zeros((n_geos, n_times))

df_grouped = df.set_index([constants.GEO, constants.TIME])
for geo_idx, geo in enumerate(geos):
    for time_idx, time in enumerate(unique_times):
        row = df_grouped.loc[(geo, time)]
        media_np[geo_idx, time_idx, :] = row[media_cols].values
        media_spend_np[geo_idx, time_idx, :] = row[media_spend_cols].values
        non_media_treatment_np[geo_idx, time_idx, :] = row[non_media_treatment_cols].values
        organic_media_np[geo_idx, time_idx, :] = row[organic_media_cols].values
        revenue_per_kpi_np[geo_idx, time_idx] = row[revenue_per_kpi_col].item()

data_tensors = analyzer.DataTensors(
    media=tf.convert_to_tensor(media_np, dtype=tf.float32),
    media_spend=tf.convert_to_tensor(media_spend_np, dtype=tf.float32),
    non_media_treatments=tf.convert_to_tensor(non_media_treatment_np, dtype=tf.float32),
    organic_media=tf.convert_to_tensor(organic_media_np, dtype=tf.float32),
    revenue_per_kpi=tf.convert_to_tensor(revenue_per_kpi_np, dtype=tf.float32),
    time=tf.convert_to_tensor(times_str, dtype=tf.string),
)
# Default values for `budget` and `pct_of_spend` are derived from `new_data`,
# but can be overridden without modifying `new_data` itself.
hypothetical_optimization_results = budget_optimizer.optimize(
    new_data=data_tensors,
    budget=50_000_000,
    pct_of_spend=[.2, .1, .2, .2, .3],
)

/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/analysis/tensors.py:702: UserWarning: A `organic_media` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/analysis/tensors.py:702: UserWarning: A `non_media_treatments` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(


Export the two-page HTML optimization report for the hypothetical scenario.

In [26]:
hypothetical_optimization_results.output_optimization_summary(
    'hypothetical_optimization_output.html', meridian_root
)

/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/analysis/tensors.py:702: UserWarning: A `organic_media` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/analysis/tensors.py:702: UserWarning: A `non_media_treatments` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(


In [27]:
IPython.display.HTML(filename=os.path.join(meridian_root, 'hypothetical_optimization_output.html'))

Channel,Non-optimized spend,Optimized spend
Channel4,30%,39%
Channel0,20%,26%
Channel2,20%,14%
Channel3,20%,14%
Channel1,10%,7%


## Step 9: Save the model object

We recommend saving the model object for future use, to avoid repetitive model runs and save time/compute. Once saved, you can reload it later to continue analysis or visualizations without re-running the model.

In [28]:
file_path = os.path.join(meridian_root, 'saved_mmm.binpb')
meridian_serde.save_meridian(mmm, file_path)
print(f'model is saved at {file_path}')

model is saved at /home/smenon01/repos/meridian-ai/demo_notebooks/meridian_output/saved_mmm.binpb


To reload the saved model:

In [29]:
mmm = meridian_serde.load_meridian(file_path)

/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/arviz/data/inference_data.py:1538: UserWarning: The group trace is not defined in the InferenceData scheme
  warnings.warn(
/home/smenon01/repos/meridian-ai/.venv/lib/python3.12/site-packages/meridian/model/knots.py:614: RuntimeWarning: overflow encountered in cast
  backend.np_float_dtype(math.comb(ncol, design_mat.shape[1]))


## Step 10: Interactive Scenario Planning

[Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner) is a tool that lets advertisers create a sharable, interactive dashboard; you can reuse the saved model from this notebook for dashboard generation.

You can also pre-generate scenario planner data files locally from your fitted model before exporting to Looker Studio.